In [1]:
import xarray as xr
import pandas as pd
import numpy as np
import math
from pathlib import Path

In [3]:
# Bioclimatic variables
bioclim_vars_path = Path("..") / "data" / "bioclimatic_variables_historic_ERA5_1980_2014.nc"
bioclim_vars = xr.open_dataset(bioclim_vars_path)

In [2]:
# Temperature
T_base_period_historic_local_path = Path("..") / "data" / "T_southamerica_base_period_historic_local.nc"
T_base_period_historic_local = xr.open_dataset(T_base_period_historic_local_path)

# Precipitation 
P_base_period_historic_local_path = Path("..") / "data" / "P_southamerica_base_period_historic_local.nc"
P_base_period_historic_local = xr.open_dataset(P_base_period_historic_local_path)

In [5]:
# Select datasets for Manaus
manaus_lat = -3.1
manaus_lon = -60.0

Tavg_manaus = T_base_period_historic_local.sel(latitude=manaus_lat, longitude=manaus_lon, method="nearest").Tavg
Pr_manaus = P_base_period_historic_local.sel(latitude=manaus_lat, longitude=manaus_lon, method="nearest").pr
bioclim_vars_manaus = bioclim_vars.sel(latitude=manaus_lat, longitude=manaus_lon, method="nearest")

In [6]:
df = pd.DataFrame(
    data=[
        Tavg_manaus.values,
        Pr_manaus.values
    ],
    index=["Temperature (°C)", "Precipitation (mm)"],
    columns=Tavg_manaus["month"].values
)

df

,1,2,3,4,5,6,7,8,9,10,11,12
Temperature (°C),26.280279,26.215279,26.332285,26.397251,26.461071,26.527391,26.617311,27.297516,27.668549,27.785126,27.415518,26.803225
Precipitation (mm),308.255506,317.620718,366.768351,364.531122,315.827192,191.391891,141.481332,125.179906,140.774242,162.245676,197.103345,278.562359


### Mean annual temperature:

Calculation: $ MAT = 1/ 12 * \sum_{i=1}^{12} T_i$ 

In [7]:
MAT_raw = 1/12 * (26.280279 + 26.215279	+ 26.332285	+ 26.397251 + 26.461071	+ 26.527391	+ 26.617311	+ 27.297516	+ 27.668549	+ 27.785126	+ 27.415518	+ 26.803225)
round(MAT_raw, 4) == round(bioclim_vars_manaus.MAT.item(), 4)

True

### Temperature seasonality

Calculation: 

Let $\bar{T}$ be annual mean temperature of that year. 

$ TS = \sqrt{1/12 \sum_{i=1}^{12}(T_i - \bar{T})^2}$

In [8]:
temp_array = df.loc["Temperature (°C)"].to_numpy()

In [10]:
t_mean = bioclim_vars_manaus.MAT.item()
sigma_squared = 0

for t in temp_array:
    sigma_squared += (t-t_mean)**2 

TS_raw = math.sqrt(1/12 * sigma_squared)

round(TS_raw, 4) == round(bioclim_vars_manaus.TS.item(), 4)

True

### Mean temperature of the Wettest Quarter

In [11]:
# First identify the quarter
prec_array = df.loc["Precipitation (mm)"].to_numpy()
sum_3 = (prec_array + np.roll(prec_array, -1) + np.roll(prec_array, -2))
idx_max_prec = np.argmax(sum_3)

# Calculate the mean temperature of quarters
temp_mean_3 = (
    temp_array
    + np.roll(temp_array, -1)
    + np.roll(temp_array, -2)
) / 3

# Select
temp_at_max_prec = temp_mean_3[idx_max_prec]
round(temp_at_max_prec, 4) == round(bioclim_vars_manaus.MTWeQ.item(), 4)


np.True_

### Mean temperature of the Driest Quarter

In [12]:
# First identify the quarter
prec_array = df.loc["Precipitation (mm)"].to_numpy()
sum_3 = (prec_array + np.roll(prec_array, -1) + np.roll(prec_array, -2))
idx_min_prec = np.argmin(sum_3)

# Calculate the mean temperature of quarters
temp_mean_3 = (
    temp_array
    + np.roll(temp_array, -1)
    + np.roll(temp_array, -2)
) / 3

# Select
temp_at_min_prec = temp_mean_3[idx_min_prec]
round(temp_at_min_prec, 4) == round(bioclim_vars_manaus.MTDQ.item(), 4)

np.True_

### Mean temperature of the Warmest Quarter

In [13]:
# First identify the quarter
temp_mean_3 = (
    temp_array
    + np.roll(temp_array, -1)
    + np.roll(temp_array, -2)
) / 3

idx_max_temp = np.argmax(temp_mean_3)

# Select
temp_at_max_temp = temp_mean_3[idx_max_temp]
round(temp_at_max_temp, 4) == round(bioclim_vars_manaus.MTWaQ.item(), 4)

np.True_

### Mean temperature of the Coldest Quarter

In [14]:
# First identify the quarter
temp_mean_3 = (
    temp_array
    + np.roll(temp_array, -1)
    + np.roll(temp_array, -2)
) / 3

idx_min_temp = np.argmin(temp_mean_3)

# Select
temp_at_min_temp = temp_mean_3[idx_min_temp]
round(temp_at_min_temp, 4) == round(bioclim_vars_manaus.MTCQ.item(), 4)

np.True_

### Annual Precipitation

In [15]:
round(prec_array.sum(), 4) == round(bioclim_vars_manaus.AP.item(), 4)

np.True_

### Precipitation of the Wettest Month

In [16]:
prec_array[prec_array.argmax()] == bioclim_vars_manaus.PWM.item()

np.True_

### Precipitation seasonality

$sdv(P) = \sqrt{1/12 \sum_{i=1}^{12}(P_i - \bar{P})^2}$ with $ \bar{P} = 1/ 12 * \sum_{i=1}^{12} P_i$ 

$PS = sdv(P) / (1+\bar{P}) * 100 $


In [17]:
p_mean = prec_array.sum()/12
p_mean

np.float64(242.47847003234554)

In [18]:
import math

sigma_squared = 0

for p in prec_array:
    sigma_squared += (p-p_mean)**2 

sdm_p = math.sqrt(1/12 * sigma_squared)

PS = sdm_p/(1+p_mean) * 100
PS 

np.float64(36.041766061435936)

In [19]:
round(PS, 4) == round(bioclim_vars_manaus.PS.item(),4)

np.True_

### Precipitation of the Wettest Quarter

In [20]:
# First identify the quarter
prec_sum_3 = (prec_array + np.roll(prec_array, -1) + np.roll(prec_array, -2))

idx_max_prec = np.argmax(prec_sum_3)

# Select
prec_at_max_prec = prec_sum_3[idx_max_prec]
round(prec_at_max_prec, 4) == round(bioclim_vars_manaus.PWeQ.item(), 4)


np.True_

### Precipitation of the Driest Quarter

In [21]:
# First identify the quarter
prec_sum_3 = (prec_array + np.roll(prec_array, -1) + np.roll(prec_array, -2))

idx_min_prec = np.argmin(prec_sum_3)

# Select
prec_at_min_prec = prec_sum_3[idx_min_prec]
round(prec_at_min_prec, 4) == round(bioclim_vars_manaus.PDQ.item(), 4)


np.True_

### Precipitaiton of the Warmest Quarter 

In [22]:
# First identify the quarter
temp_mean_3 = (
    temp_array
    + np.roll(temp_array, -1)
    + np.roll(temp_array, -2)
) / 3

idx_max_temp = np.argmax(temp_mean_3)


prec_sum_3 = (prec_array + np.roll(prec_array, -1) + np.roll(prec_array, -2))

# Select
prec_at_max_temp = prec_sum_3[idx_max_temp]
round(prec_at_max_temp, 4) == round(bioclim_vars_manaus.PWaQ.item(), 4)

np.True_

### Precipitation of the coldest quarter

In [23]:
# First identify the quarter
temp_mean_3 = (
    temp_array
    + np.roll(temp_array, -1)
    + np.roll(temp_array, -2)
) / 3

idx_min_temp = np.argmin(temp_mean_3)


prec_sum_3 = (prec_array + np.roll(prec_array, -1) + np.roll(prec_array, -2))

# Select
prec_at_min_temp = prec_sum_3[idx_min_temp]
round(prec_at_min_temp, 4) == round(bioclim_vars_manaus.PCQ.item(), 4)

np.True_